<h1> GOterm Annotation </h1> 
In this notebook, I'll annotate the CpG background and DML lists from methylKit and DSS with GOterms. I will use these annotations for gene enrichment.

Create master annotation table
Match CpG background and DML lists with GOterms
Modify lists for downstream gene enrichment

<h2> 0. Prepare to run script </h2> 
<h3> 0a. Set working directory </h3>

In [22]:
pwd

'/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS'

In [2]:
# import dill
# dill.dump_session('JupyterLab_sessions/20210624_GOterm_notebook.db')

In [23]:
# cd analyses/

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/analyses


In [6]:
#!mkdir GOterm-annotation/

<h2> Create master annotation table </h2>
<h3> 1a. Format DIAMOND blastx output </h3>



In [16]:
pwd

'/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/analyses/GOterm-annotation/BOTH'

In [17]:
cd /home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/blast

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/blast


In [18]:
#Download blastx output
!wget https://gannet.fish.washington.edu/spartina/project-gigas-oa-meth/output/blastx/20210605-cgigas-roslin-mito-blastx.outfmt6 \
--no-check-certificate

--2021-11-24 00:31:21--  https://gannet.fish.washington.edu/spartina/project-gigas-oa-meth/output/blastx/20210605-cgigas-roslin-mito-blastx.outfmt6
Resolving gannet.fish.washington.edu (gannet.fish.washington.edu)... 128.95.149.52
Connecting to gannet.fish.washington.edu (gannet.fish.washington.edu)|128.95.149.52|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 10442376 (10.0M)
Saving to: ‘20210605-cgigas-roslin-mito-blastx.outfmt6’

20210605-cgigas-ros 100%[===================>]   9.96M  22.6MB/s    in 0.4s    

2021-11-24 00:31:22 (22.6 MB/s) - ‘20210605-cgigas-roslin-mito-blastx.outfmt6’ saved [10442376/10442376]



In [19]:
#Check output
#!head 20210605-cgigas-roslin-mito-blastx.outfmt6
#!wc -l 20210605-cgigas-roslin-mito-blastx.outfmt6

#convert pipes to tab to isolate Uniprot accession code
!tr '|' '\t' < 20210605-cgigas-roslin-mito-blastx.outfmt6 \
> cgigas-roslin-mito-blastx.outfmt6.codeIsolated

#!head cgigas-roslin-mito-blastx.outfmt6.codeIsolated
#!wc -l cgigas-roslin-mito-blastx.outfmt6.codeIsolated

In [20]:
#Extract column with transcript ID
#Convert _ to tab
#Extract column with transcript ID
#Add XM_ to the front of each ID

!cut -f2 cgigas-roslin-mito-blastx.outfmt6.codeIsolated \
| tr "_" "\t" \
| cut -f5 \
| sed 's/^/XM_/' \
> cgigas-roslin-mito-blastx.outfmt6.transcriptID

#!head cgigas-roslin-mito-blastx.outfmt6.transcriptID
#!wc -l cgigas-roslin-mito-blastx.outfmt6.transcriptID

In [21]:
#Paste original transcript ID with codeIsolated file
#Check output

!paste cgigas-roslin-mito-blastx.outfmt6.transcriptID cgigas-roslin-mito-blastx.outfmt6.codeIsolated \
> cgigas-roslin-mito-blastx.outfmt6.codeIsolated.transcriptID
!head cgigas-roslin-mito-blastx.outfmt6.codeIsolated.transcriptID

XM_034463066.1	lcl	NC_047559.1_mrna_XM_034463066.1_26	sp	A1KZ92	PXDNL_HUMAN	24.891	229	153	6	867	1535	278	493	4.55e-05	51.2
XM_034463224.1	lcl	NC_047559.1_mrna_XM_034463224.1_35	sp	Q7LHG5	YI31B_YEAST	50.658	152	71	1	1072	1515	617	768	8.84e-40	157
XM_034456887.1	lcl	NC_047559.1_mrna_XM_034456887.1_39	sp	A1A5V9	ELP5_DANRE	28.873	284	194	4	356	1198	1	279	6.43e-29	118
XM_011455176.3	lcl	NC_047559.1_mrna_XM_011455176.3_47	sp	A1A5V9	ELP5_DANRE	28.873	284	194	4	376	1218	1	279	7.04e-29	118
XM_011455177.3	lcl	NC_047559.1_mrna_XM_011455177.3_48	sp	A1A5V9	ELP5_DANRE	28.873	284	194	4	245	1087	1	279	3.80e-29	118
XM_011419438.3	lcl	NC_047559.1_mrna_XM_011419438.3_53	sp	A0A159BP93	CITB_MONRU	30.795	302	179	11	3053	2199	3	291	3.63e-29	123
XM_020064467.2	lcl	NC_047559.1_mrna_XM_020064467.2_54	sp	A0A159BP93	CITB_MONRU	30.795	302	179	11	3307	2453	3	291	3.97e-29	123
XM_011419437.3	lcl	NC_047559.1_mrna_XM_011419437.3_55	sp	O35077	GPDA_RAT	59.259	351	140	3	111	1160	1	349	4.39e-140	434
XM_011419460.2	lcl	NC_

In [22]:
#Reduce the number of columns using awk: accession code, transcript ID, and e-value
#Sort, and save as a new file.
!awk -v OFS='\t' '{print $5, $1, $15}' < cgigas-roslin-mito-blastx.outfmt6.codeIsolated.transcriptID | sort \
> gigas-blast-sort.tab

In [23]:
!head gigas-blast-sort.tab

A0A060WQA3	XM_011455784.3	4.79e-06
A0A060WQA3	XM_011455785.3	4.65e-06
A0A061ACU2	XM_034483499.1	0.0
A0A061ACU2	XM_034483499.1	1.40e-12
A0A061ACU2	XM_034483499.1	2.13e-27
A0A061ACU2	XM_034483500.1	0.0
A0A061ACU2	XM_034483500.1	1.40e-12
A0A061ACU2	XM_034483500.1	2.44e-27
A0A061ACU2	XM_034483501.1	0.0
A0A061ACU2	XM_034483501.1	1.40e-12


<h3> 1b. Join with GOterms </h3>

In [24]:
#Download Uniprot database with GOterm information
!curl -O http://owl.fish.washington.edu/halfshell/bu-alanine-wd/17-07-20/uniprot-SP-GO.sorted

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  340M  100  340M    0     0  18.4M      0  0:00:18  0:00:18 --:--:-- 30.1M


In [25]:
!head uniprot-SP-GO.sorted
!wc -l uniprot-SP-GO.sorted

A0A023GPI8	LECA_CANBL	reviewed	Lectin alpha chain (CboL) [Cleaved into: Lectin beta chain; Lectin gamma chain]		Canavalia boliviana	237			mannose binding [GO:0005537]; metal ion binding [GO:0046872]	mannose binding [GO:0005537]; metal ion binding [GO:0046872]	GO:0005537; GO:0046872
A0A023GPJ0	CDII_ENTCC	reviewed	Immunity protein CdiI	cdiI ECL_04450.1	Enterobacter cloacae subsp. cloacae (strain ATCC 13047 / DSM 30054 / NBRC 13535 / NCDC 279-56)	145					
A0A023PXA5	YA19A_YEAST	reviewed	Putative uncharacterized protein YAL019W-A	YAL019W-A	Saccharomyces cerevisiae (strain ATCC 204508 / S288c) (Baker's yeast)	189					
A0A023PXB0	YA019_YEAST	reviewed	Putative uncharacterized protein YAR019W-A	YAR019W-A	Saccharomyces cerevisiae (strain ATCC 204508 / S288c) (Baker's yeast)	110					
A0A023PXB5	IRC2_YEAST	reviewed	Putative uncharacterized membrane protein IRC2 (Increased recombination centers protein 2)	IRC2 YDR112W	Saccharomyces cerevisiae (strain ATCC 204508 / S288c) (Baker's yeast)	102		integ

In [26]:
#Join the first column in the first file with the first column in the second file
#The files are tab delimited, and the output should also be tab delimited (-t $'\t')
!join -1 1 -2 1 -t $'\t' \
gigas-blast-sort.tab \
uniprot-SP-GO.sorted \
> gigas-blast-annot.tab
!head gigas-blast-annot.tab
!wc -l gigas-blast-annot.tab

A0A0A7DNP6	XM_001305294.1	8.75e-10	GRHLP_RUDPH	reviewed	Prepro-gonadotropin-releasing hormone-like protein (rp-GnRH) [Cleaved into: GnRH dodecapeptide; GnRH-associated peptide (GAP)]		Ruditapes philippinarum (Japanese littleneck clam) (Venerupis philippinarum)	94	neuropeptide signaling pathway [GO:0007218]	extracellular region [GO:0005576]		extracellular region [GO:0005576]; neuropeptide signaling pathway [GO:0007218]	GO:0005576; GO:0007218
A0A0B5A7M7	XM_001308866.2	3.97e-12	INS1_CONIM	reviewed	Con-Ins Im1 (Insulin 1) [Cleaved into: Con-Ins I1 B chain; Con-Ins I1 A chain]		Conus imperialis (Imperial cone)	150	glucose metabolic process [GO:0006006]	extracellular region [GO:0005576]	hormone activity [GO:0005179]	extracellular region [GO:0005576]; hormone activity [GO:0005179]; glucose metabolic process [GO:0006006]	GO:0005179; GO:0005576; GO:0006006
A0A0B5A7M7	XM_034475035.1	1.48e-12	INS1_CONIM	reviewed	Con-Ins Im1 (Insulin 1) [Cleaved into: Con-Ins I1 B chain; Con-Ins I1 A chain]		Conus

In [27]:
#Extract columns 1 (accession), 2 (transcript ID), and 14 (GOterms)
#Save output
!cut -f1,2,14 gigas-blast-annot.tab \
> _blast-annot.tab
!head _blast-annot.tab

A0A0A7DNP6	XM_001305294.1	GO:0005576; GO:0007218
A0A0B5A7M7	XM_001308866.2	GO:0005179; GO:0005576; GO:0006006
A0A0B5A7M7	XM_034475035.1	GO:0005179; GO:0005576; GO:0006006
A0A0B5A8P8	XM_011417422.3	GO:0005179; GO:0005576; GO:0006006
A0A0F7YYX3	XM_011419035.3	GO:0005576
A0A0F7YYX3	XM_011419037.3	GO:0005576
A0A0F7YYX3	XM_011419038.3	GO:0005576
A0A0F7YYX3	XM_011419039.3	GO:0005576
A0A0F7YYX3	XM_020064360.2	GO:0005576
A0A0F7YZI5	XM_011442798.3	GO:0005179; GO:0005576


In [28]:
%%bash 

# This script was originally written to address a specific problem that Rhonda was having

# input_file is the initial, "problem" file
# file is an intermediate file that most of the program works upon
# output_file is the final file produced by the script
input_file="_blast-annot.tab"
file="_intermediate.file"
output_file="_blast-GO-unfolded.tab"

# sed command substitutes the "; " sequence to a tab and writes the new format to a new file.
# This character sequence is how the GO terms are delimited in their field.
sed $'s/; /\t/g' "$input_file" > "$file"

# Identify first field containing a GO term.
# Search file with grep for "GO:" and pipe to awk.
# Awk sets tab as field delimiter (-F'\t'), runs a for loop that looks for "GO:" (~/GO:/), and then prints the field number).
# Awk results are piped to sort, which sorts unique by number (-ug).
# Sort results are piped to head to retrieve the lowest value (i.e. the top of the list; "-n1").
begin_goterms=$(grep "GO:" "$file" | awk -F'\t' '{for (i=1;i<=NF;i++) if($i ~/GO:/) print i}' | sort -ug | head -n1)

# While loop to process each line of the input file.
while read -r line
	do
	
	# Send contents of the current line to awk.
	# Set the field separator as a tab (-F'\t') and print the number of fields in that line.
	# Save the results of the echo/awk pipe (i.e. number of fields) to the variable "max_field".
	max_field=$(echo "$line" | awk -F'\t' '{print NF}')

	# Send contents of current line to cut.
	# Cut fields (i.e. retain those fields) 1-12.
	# Save the results of the echo/cut pipe (i.e. fields 1-12) to the variable "fixed_fields"
	fixed_fields=$(echo "$line" | cut -f1-2)

	# Since not all the lines contain the same number of fields (e.g. may not have GO terms),
	# evaluate the number of fields in each line to determine how to handle current line.

	# If the value in max_field is less than the field number where the GO terms begin,
	# then just print the current line (%s) followed by a newline (\n).
	if (( "$max_field" < "$begin_goterms" ))
		then printf "%s\n" "$line"
			else

			# Send contents of current line (which contains GO terms) to cut.
			# Cut fields (i.e. retain those fields) 13 to whatever the last field is in the curent line.
			# Save the results of the echo/cut pipe (i.e. all the GO terms fields) to the variable "goterms".
			goterms=$(echo "$line" | cut -f"$begin_goterms"-"$max_field")
			
			# Assign values in the variable "goterms" to a new indexed array (called "array"), 
			# with tab delimiter (IFS=$'\t')
			IFS=$'\t' read -r -a array <<<"$goterms"
			
			# Iterate through each element of the array.
			# Print the first 12 fields (i.e. the fields stored in "fixed_fields") followed by a tab (%s\t).
			# Print the current element in the array (i.e. the current GO term) followed by a new line (%s\n).
			for element in "${!array[@]}"	
				do printf "%s\t%s\n" "$fixed_fields" "${array[$element]}"
			done
	fi

# Send the input file into the while loop and send the output to a file named "rhonda_fixed.txt".
done < "$file" > "$output_file"

Process is interrupted.


In [30]:
!head _blast-GO-unfolded.tab

A0A0A7DNP6	XM_001305294.1	GO:0005576
A0A0A7DNP6	XM_001305294.1	GO:0007218
A0A0B5A7M7	XM_001308866.2	GO:0005179
A0A0B5A7M7	XM_001308866.2	GO:0005576
A0A0B5A7M7	XM_001308866.2	GO:0006006
A0A0B5A7M7	XM_034475035.1	GO:0005179
A0A0B5A7M7	XM_034475035.1	GO:0005576
A0A0B5A7M7	XM_034475035.1	GO:0006006
A0A0B5A8P8	XM_011417422.3	GO:0005179
A0A0B5A8P8	XM_011417422.3	GO:0005576


In [9]:
#Reorganize and sort columns
!awk '{print $3"\t"$2}' _blast-GO-unfolded.tab | sort -V > _blast-GO-unfolded.sorted

In [10]:
!head _blast-GO-unfolded.sorted
!wc _blast-GO-unfolded.sorted

GO:0000002	XM_004596953.1
GO:0000002	XM_004599087.1
GO:0000002	XM_004599974.1
GO:0000002	XM_004602618.1
GO:0000002	XM_004604080.1
GO:0000002	XM_011413926.3
GO:0000002	XM_011416774.2
GO:0000002	XM_011428102.3
GO:0000002	XM_011430231.3
GO:0000002	XM_011434743.3
 1565846  3129025 40681933 _blast-GO-unfolded.sorted


<h3> 1c. Join with GO Slim terms </h3>

In [11]:
#Get GO to GOSlim matching
!curl -O http://owl.fish.washington.edu/halfshell/bu-alanine-wd/17-07-20/GO-GOslim.sorted

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2314k  100 2314k    0     0  2849k      0 --:--:-- --:--:-- --:--:-- 2846k


In [12]:
!head GO-GOslim.sorted
!wc -l GO-GOslim.sorted

GO:0000001	mitochondrion inheritance	cell organization and biogenesis	P
GO:0000002	mitochondrial genome maintenance	cell organization and biogenesis	P
GO:0000003	reproduction	other biological processes	P
GO:0000006	high affinity zinc uptake transmembrane transporter activity	transporter activity	F
GO:0000007	low-affinity zinc ion transmembrane transporter activity	transporter activity	F
GO:0000009	"alpha-1,6-mannosyltransferase activity"	other molecular function	F
GO:0000010	trans-hexaprenyltranstransferase activity	other molecular function	F
GO:0000011	vacuole inheritance	cell organization and biogenesis	P
GO:0000012	single strand break repair	DNA metabolism	P
GO:0000012	single strand break repair	stress response	P
30796 GO-GOslim.sorted


In [13]:
#Join files to get GOslim for each query (with duplicate GOslim / query removed)
!join -1 1 -2 1 -t $'\t' \
_blast-GO-unfolded.sorted \
GO-GOslim.sorted \
| uniq | awk -F'\t' -v OFS='\t' '{print $2, $1, $3, $4, $5}' \
| sort \
> Blastquery-GOslim.tab
!head Blastquery-GOslim.tab
!wc -l Blastquery-GOslim.tab

join: _blast-GO-unfolded.sorted:1563180: is not sorted: 	XM_899336.3
XM_	GO:0016021	integral to membrane	other membranes	C
XM_001305288.1	GO:0001501	skeletal system development	developmental processes	P
XM_001305288.1	GO:0005576	extracellular region	non-structural extracellular	C
XM_001305288.1	GO:0005595	collagen type XII	extracellular matrix	C
XM_001305288.1	GO:0005615	extracellular space	non-structural extracellular	C
XM_001305288.1	GO:0005788	endoplasmic reticulum lumen	ER/Golgi	C
XM_001305288.1	GO:0007155	cell adhesion	cell adhesion	P
XM_001305288.1	GO:0030020	extracellular matrix structural constituent conferring tensile strength	extracellular structural activity	F
XM_001305288.1	GO:0030199	collagen fibril organization	cell organization and biogenesis	P
XM_001305288.1	GO:0030574	collagen catabolic process	other metabolic processes	P
754502 Blastquery-GOslim.tab


<h2> 2. Match CpG background and DML lists with GOterms </h2>

<h3> 2a. Obtain transcript and gene ID information from mRNA track </h3>

In [6]:
pwd

'/home/mngeorge/project-gigas_ploidy'

In [8]:
cd bisulfide_analysis/WGBS/

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS


In [9]:
!head GFF/cgigas_uk_roslin_v1_mRNA.gff
!wc -l GFF/cgigas_uk_roslin_v1_mRNA.gff

NC_047559.1	Gnomon	mRNA	14114	15804	.	+	.	ID=rna-XM_034463183.1;Parent=gene-LOC109621113;Dbxref=GeneID:109621113,Genbank:XM_034463183.1;Name=XM_034463183.1;gbkey=mRNA;gene=LOC109621113;model_evidence=Supporting evidence includes similarity to: 78%25 coverage of the annotated genomic feature by RNAseq alignments%2C including 21 samples with support for all annotated introns;product=uncharacterized LOC109621113;transcript_id=XM_034463183.1
NC_047559.1	Gnomon	mRNA	16867	19160	.	-	.	ID=rna-XM_034463195.1;Parent=gene-LOC117687066;Dbxref=GeneID:117687066,Genbank:XM_034463195.1;Name=XM_034463195.1;gbkey=mRNA;gene=LOC117687066;model_evidence=Supporting evidence includes similarity to: 98%25 coverage of the annotated genomic feature by RNAseq alignments%2C including 14 samples with support for all annotated introns;product=uncharacterized LOC117687066;transcript_id=XM_034463195.1
NC_047559.1	Gnomon	mRNA	61887	71038	.	-	.	ID=rna-XM_034471753.1;Parent=gene-LOC117689737;Dbxref=GeneID:117689737,Gen

In [10]:
#Isolate column with ID information
#Convert multiple delimiters to tabs
#Isolate transcript and gene ID columns
#Ensure they are tab-delimited
#Sort by gene ID
#Save output

!cut -f9 GFF/cgigas_uk_roslin_v1_mRNA.gff \
| tr "=;:,-" "\t" \
| cut -f3,9 \
| awk '{print $2"\t"$1}' \
| sort \
> analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID

In [27]:
!head analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID
!wc -l analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID

105317035	XM_011413559.3
105317035	XM_034443160.1
105317035	XM_034443162.1
105317035	XM_034443163.1
105317035	XM_034443164.1
105317036	XM_034443166.1
105317036	XM_034443167.1
105317036	XM_034443168.1
105317040	XM_011413566.3
105317040	XM_034443178.1
63341 analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID


*****************************************
<h3> 2b. ploidy-DML </h3>

In [27]:
cd /home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS


In [3]:
!intersectBed \
-wb \
-a DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-PLOIDY.bed \
-b GFF/cgigas_uk_roslin_v1_gene.gff \
> DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-PLOIDY-Gene-wb.bed
!tail DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-PLOIDY-Gene-wb.bed

NW_022994997.1	204573	204575	-21.28520926	NW_022994997.1	Gnomon	gene	166081	209461	.	+	.	ID=gene-LOC105342885;Dbxref=GeneID:105342885;Name=LOC105342885;gbkey=Gene;gene=LOC105342885;gene_biotype=protein_coding
NW_022994997.1	204573	204575	-21.28520926	NW_022994997.1	Gnomon	gene	166519	233361	.	-	.	ID=gene-LOC105342900;Dbxref=GeneID:105342900;Name=LOC105342900;gbkey=Gene;gene=LOC105342900;gene_biotype=protein_coding
NW_022994997.1	208908	208910	-24.80703921	NW_022994997.1	Gnomon	gene	166081	209461	.	+	.	ID=gene-LOC105342885;Dbxref=GeneID:105342885;Name=LOC105342885;gbkey=Gene;gene=LOC105342885;gene_biotype=protein_coding
NW_022994997.1	208908	208910	-24.80703921	NW_022994997.1	Gnomon	gene	166519	233361	.	-	.	ID=gene-LOC105342900;Dbxref=GeneID:105342900;Name=LOC105342900;gbkey=Gene;gene=LOC105342900;gene_biotype=protein_coding
NW_022994998.1	1293	1295	36.28232549	NW_022994998.1	Gnomon	gene	1077	17682	.	+	.	ID=gene-LOC105321748;Dbxref=GeneID:105321748;Name=LOC105321748;gbkey=Gene;gene=LOC1

In [4]:
!intersectBed \
-wb \
-a DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-HEAT.bed \
-b GFF/cgigas_uk_roslin_v1_gene.gff \
> DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-HEAT-Gene-wb.bed
!tail DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-HEAT-Gene-wb.bed

NW_022994987.1	149169	149171	-23.847750109218	NW_022994987.1	Gnomon	gene	123012	150079	.	+	.	ID=gene-LOC105346443;Dbxref=GeneID:105346443;Name=LOC105346443;gbkey=Gene;gene=LOC105346443;gene_biotype=protein_coding
NW_022994988.1	7697	7699	-29.7786892249851	NW_022994988.1	Gnomon	gene	2092	25134	.	-	.	ID=gene-LOC105327367;Dbxref=GeneID:105327367;Name=LOC105327367;end_range=25134,.;gbkey=Gene;gene=LOC105327367;gene_biotype=protein_coding;partial=true
NW_022994989.1	53683	53685	-27.2194135490394	NW_022994989.1	Gnomon	gene	38643	64729	.	-	.	ID=gene-LOC105345846;Dbxref=GeneID:105345846;Name=LOC105345846;gbkey=Gene;gene=LOC105345846;gene_biotype=protein_coding
NW_022994989.1	53707	53709	-27.582787167989	NW_022994989.1	Gnomon	gene	38643	64729	.	-	.	ID=gene-LOC105345846;Dbxref=GeneID:105345846;Name=LOC105345846;gbkey=Gene;gene=LOC105345846;gene_biotype=protein_coding
NW_022994989.1	63311	63313	20.1545893719807	NW_022994989.1	Gnomon	gene	38643	64729	.	-	.	ID=gene-LOC105345846;Dbxref=GeneID:105345

In [5]:
!intersectBed \
-wb \
-a DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-COMBO.bed \
-b GFF/cgigas_uk_roslin_v1_gene.gff \
> DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-COMBO-Gene-wb.bed
!tail DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-COMBO-Gene-wb.bed

NW_022994916.1	48211	48213	-28.17512275	NW_022994916.1	Gnomon	gene	48124	63862	.	-	.	ID=gene-LOC109619669;Dbxref=GeneID:109619669;Name=LOC109619669;gbkey=Gene;gene=LOC109619669;gene_biotype=protein_coding
NW_022994918.1	340314	340316	-22.06142698	NW_022994918.1	Gnomon	gene	335598	404469	.	-	.	ID=gene-LOC105332697;Dbxref=GeneID:105332697;Name=LOC105332697;end_range=404469,.;gbkey=Gene;gene=LOC105332697;gene_biotype=protein_coding;partial=true
NW_022994918.1	396187	396189	27.01662562	NW_022994918.1	Gnomon	gene	335598	404469	.	-	.	ID=gene-LOC105332697;Dbxref=GeneID:105332697;Name=LOC105332697;end_range=404469,.;gbkey=Gene;gene=LOC105332697;gene_biotype=protein_coding;partial=true
NW_022994932.1	111208	111210	23.3244599	NW_022994932.1	Gnomon	gene	67831	123306	.	-	.	ID=gene-LOC105326047;Dbxref=GeneID:105326047;Name=LOC105326047;end_range=123306,.;gbkey=Gene;gene=LOC105326047;gene_biotype=protein_coding;partial=true
NW_022994948.1	5613	5615	-20.35897436	NW_022994948.1	Gnomon	gene	318	28612	.

In [6]:
#Isolate column with gene ID information
#Convert =, :, and ; to \t
#Isolate gene ID

!cut -f13 DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-PLOIDY-Gene-wb.bed \
| tr "=:;" "\t" \
| cut -f5 \
> DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs
!wc -l DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs

!cut -f13 DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-HEAT-Gene-wb.bed \
| tr "=:;" "\t" \
| cut -f5 \
> DML/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs
!wc -l DML/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs

!cut -f13 DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-COMBO-Gene-wb.bed \
| tr "=:;" "\t" \
| cut -f5 \
> DML/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs
!wc -l DML/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs

1122 DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs
2370 DML/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs
274 DML/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs


In [7]:
!paste DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-PLOIDY-Gene-wb.bed \
| awk -F'\t' -v OFS='\t' '{print $1, $2, $3, $4, $5}' \
| sort \
> DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap
!wc -l DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap

!paste DML/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-HEAT-Gene-wb.bed \
| awk -F'\t' -v OFS='\t' '{print $1, $2, $3, $4, $5}' \
| sort \
> DML/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap
!wc -l DML/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap

!paste DML/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs DML/BOTH/DML-getMethylDiff-Cov10-20-BOTH-COMBO-Gene-wb.bed \
| awk -F'\t' -v OFS='\t' '{print $1, $2, $3, $4, $5}' \
| sort \
> DML/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap
!wc -l DML/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap

1122 DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap
2370 DML/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap
274 DML/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap


<h3> Join with transcript IDs </h3>

In [9]:
#Join files to get transcript ID for DML
!join -1 1 -2 1 -t $'\t' \
DML/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap \
analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID \
| uniq | awk -F'\t' -v OFS='\t' '{print $6, $1, $2, $3, $4, $5}' \
| sort \
> analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap.transcriptIDs

#Col names: transcript IDs, gene IDs, chr, start, end, meth.diff
#!head analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap.transcriptIDs
!wc -l analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap.transcriptIDs

!join -1 1 -2 1 -t $'\t' \
DML/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap \
analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID \
| uniq | awk -F'\t' -v OFS='\t' '{print $6, $1, $2, $3, $4, $5}' \
| sort \
> analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap.transcriptIDs

#Col names: transcript IDs, gene IDs, chr, start, end, meth.diff
#!head analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap.transcriptIDs
!wc -l analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap.transcriptIDs

!join -1 1 -2 1 -t $'\t' \
DML/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap \
analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID \
| uniq | awk -F'\t' -v OFS='\t' '{print $6, $1, $2, $3, $4, $5}' \
| sort \
> analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap.transcriptIDs

#Col names: transcript IDs, gene IDs, chr, start, end, meth.diff
#!head analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap.transcriptIDs
!wc -l analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap.transcriptIDs

3695 analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap.transcriptIDs
8132 analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap.transcriptIDs
545 analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap.transcriptIDs


<h3> Join with annotations </h3>

In [10]:
#Join files to get GO annotations for DML
!join -1 1 -2 1 -t $'\t' \
analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.geneOverlap.transcriptIDs \
analyses/GOterm-annotation/Blastquery-GOslim.tab \
| uniq \
| sort \
> analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

#!head analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs.geneOverlap.transcriptIDs.GOAnnot
!wc -l analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

!join -1 1 -2 1 -t $'\t' \
analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.geneOverlap.transcriptIDs \
analyses/GOterm-annotation/Blastquery-GOslim.tab \
| uniq \
| sort \
> analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

#!head analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs.geneOverlap.transcriptIDs.GOAnnot
!wc -l analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

!join -1 1 -2 1 -t $'\t' \
analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.geneOverlap.transcriptIDs \
analyses/GOterm-annotation/Blastquery-GOslim.tab \
| uniq \
| sort \
> analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

#!head analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs.geneOverlap.transcriptIDs.GOAnnot
!wc -l analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

46626 analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-PLOIDY.GeneIDs.geneOverlap.transcriptIDs.GOAnnot
106621 analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-HEAT.GeneIDs.geneOverlap.transcriptIDs.GOAnnot
7029 analyses/GOterm-annotation/BOTH/DML-Cov10-20-BOTH-COMBO.GeneIDs.geneOverlap.transcriptIDs.GOAnnot


<h2> 2c. CpG background (1x CpGs) </h2>

In [17]:
#Install pandas for this notebook
import pandas as pd
print(pd.__version__)

1.2.4


In [18]:
pwd

'/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS'

In [ ]:
#Download files
!wget -r \
--no-check-certificate --no-directories --no-parent --reject "index.html*" \
-P . \
-A "*merged_CpG_evidence.cov" https://gannet.fish.washington.edu/panopea/030521-ronrosM/

In [31]:
#Download files
!wget -r \
--no-check-certificate --no-directories --no-parent --reject "index.html*" \
-P . \
-A "*zr3644_7_R1.CpG_report.merged_CpG_evidence.cov" https://gannet.fish.washington.edu/panopea/061021-big/

--2021-11-15 14:02:08--  https://gannet.fish.washington.edu/panopea/061021-big/
Resolving gannet.fish.washington.edu (gannet.fish.washington.edu)... 128.95.149.52
Connecting to gannet.fish.washington.edu (gannet.fish.washington.edu)|128.95.149.52|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘./index.html.tmp’

index.html.tmp          [ <=>                ] 342.01K  --.-KB/s    in 0.03s   

2021-11-15 14:02:20 (12.4 MB/s) - ‘./index.html.tmp’ saved [350223]

Loading robots.txt; please ignore errors.
--2021-11-15 14:02:20--  https://gannet.fish.washington.edu/robots.txt
Reusing existing connection to gannet.fish.washington.edu:443.
HTTP request sent, awaiting response... 404 Not Found
2021-11-15 14:02:20 ERROR 404: Not Found.

Removing ./index.html.tmp since it should be rejected.

--2021-11-15 14:02:20--  https://gannet.fish.washington.edu/panopea/061021-big/?C=N;O=D
Reusin

In [19]:
cd COVfiles/BOTH/

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/COVfiles/BOTH


In [20]:
#Check directory for all files
!ls -lh

total 4.3G
-rw-r--r-- 1 mngeorge mngeorge 441M Nov 15 14:17 zr3534_10_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 428M Nov 15 14:16 zr3534_1_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 428M Nov 15 14:17 zr3534_2_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 433M Nov 15 14:17 zr3534_3_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 428M Nov 15 14:17 zr3534_4_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 432M Nov 15 14:17 zr3534_5_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 442M Nov 15 14:17 zr3534_6_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 434M Nov 15 14:17 zr3534_7_R1.fastp-trim.20201202.CpG_report.merged_CpG_evidence.cov
-rw-r--r-- 1 mngeorge mngeorge 446M Nov 15 14:18 zr3534_8_R1

In [21]:
%%bash

for f in *.cov
do
    awk '{print $1"\t"$2"\t"$3"\t"$4}' ${f} \
    > $(basename ${f%..CpG_report.merged_CpG_evidence.cov})_1x.bedgraph
done

Process is interrupted.


In [40]:
%%bash

for f in *1x.bedgraph
do
    sortBed \
    -i ${f} \
    > $(basename ${f%_1x.bedgraph})_1x.sort.bedgraph
done

In [22]:
cd /home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/BEDgraphs/BOTH

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/BEDgraphs/BOTH


In [24]:
#Create union BEDgraph from sorted files
#Include a header
#Use N/A when there is no data for a CpG in a sample
#Define sample IDs
#Use sorted bedgraphs
#Save output
!unionBedGraphs \
-header \
-filler N/A \
-names 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 \
-i \
*1x.sort.bedgraph \
> /home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/BEDgraphs/BOTH/union_1x.BOTH.bedgraph

^C


In [28]:
cd /home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS


In [25]:
#Convert 1x bedgraph (DSS CpG background) to BEDfile. 
#Remove header
#Save output: Will intersect with genes to get information important for annotation
!awk '{print $1"\t"$2"\t"$3}' BEDgraphs/BOTH/union_1x.BOTH.bedgraph \
| tail -n+2 \
> BEDgraphs/BOTH/union_1x.BOTH.bedgraph.bed
!head BEDgraphs/BOTH/union_1x.BOTH.bedgraph.bed

/bin/bash: BEDgraphs/BOTH/union_1x.BOTH.bedgraph.bed: No such file or directory
awk: fatal: cannot open file `BEDgraphs/BOTH/union_1x.BOTH.bedgraph' for reading (No such file or directory)
head: cannot open 'BEDgraphs/BOTH/union_1x.BOTH.bedgraph.bed' for reading: No such file or directory


In [57]:
#Characterize overlap for downstream annotation
!intersectBed \
-wb \
-a BEDgraphs/BOTH/union_1x.BOTH.bedgraph.bed \
-b GFF/cgigas_uk_roslin_v1_gene.gff \
> BEDgraphs/BOTH/union_1x-BOTH-Gene-wb.bed
!head BEDgraphs/BOTH/union_1x-BOTH-Gene-wb.bed

NC_047559.1	10155	10157	NC_047559.1	Gnomon	gene	9839	11386	.	+	.	ID=gene-LOC117693020;Dbxref=GeneID:117693020;Name=LOC117693020;gbkey=Gene;gene=LOC117693020;gene_biotype=lncRNA
NC_047559.1	10215	10217	NC_047559.1	Gnomon	gene	9839	11386	.	+	.	ID=gene-LOC117693020;Dbxref=GeneID:117693020;Name=LOC117693020;gbkey=Gene;gene=LOC117693020;gene_biotype=lncRNA
NC_047559.1	10270	10272	NC_047559.1	Gnomon	gene	9839	11386	.	+	.	ID=gene-LOC117693020;Dbxref=GeneID:117693020;Name=LOC117693020;gbkey=Gene;gene=LOC117693020;gene_biotype=lncRNA
NC_047559.1	10292	10294	NC_047559.1	Gnomon	gene	9839	11386	.	+	.	ID=gene-LOC117693020;Dbxref=GeneID:117693020;Name=LOC117693020;gbkey=Gene;gene=LOC117693020;gene_biotype=lncRNA
NC_047559.1	10314	10316	NC_047559.1	Gnomon	gene	9839	11386	.	+	.	ID=gene-LOC117693020;Dbxref=GeneID:117693020;Name=LOC117693020;gbkey=Gene;gene=LOC117693020;gene_biotype=lncRNA
NC_047559.1	10358	10360	NC_047559.1	Gnomon	gene	9839	11386	.	+	.	ID=gene-LOC117693020;Dbxref=GeneID:117693020;Name=

In [58]:
pwd

'/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS'

In [60]:
#Isolate column with gene ID information
#Convert =, :, and ; to \t
#Isolate gene ID

!cut -f12 BEDgraphs/BOTH/union_1x-BOTH-Gene-wb.bed \
| tr "=:;" "\t" \
| cut -f5 \
> analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs
!head analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs
!wc -l analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs

117693020
117693020
117693020
117693020
117693020
117693020
117693020
117693020
117693020
117693020
7747915 analyses/GOterm-annotation/YAAMINI/union_1x-YAAMINI.GeneIDs


In [29]:
pwd

'/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS'

In [30]:
!paste analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs BEDgraphs/BOTH/union_1x-Gene-wb.bed \
| awk -F'\t' -v OFS='\t' '{print $1, $2, $3, $4, $5}' \
| sort \
> analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap
!head analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap
!wc -l analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap

105317035	NC_047564.1	46227067	46227069	NC_047564.1
105317035	NC_047564.1	46227126	46227128	NC_047564.1
105317035	NC_047564.1	46227330	46227332	NC_047564.1
105317035	NC_047564.1	46227355	46227357	NC_047564.1
105317035	NC_047564.1	46227472	46227474	NC_047564.1
105317035	NC_047564.1	46227513	46227515	NC_047564.1
105317035	NC_047564.1	46227569	46227571	NC_047564.1
105317035	NC_047564.1	46227587	46227589	NC_047564.1
105317035	NC_047564.1	46227597	46227599	NC_047564.1
105317035	NC_047564.1	46227606	46227608	NC_047564.1
7913468 analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap


<h3> Join with transcript IDs </h3>

In [31]:
#Join files to get transcript ID for DML
!join -1 1 -2 1 -t $'\t' \
analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap \
analyses/GOterm-annotation/cgigas_uk_roslin_v1_mRNA.transcriptID.geneID \
| uniq | awk -F'\t' -v OFS='\t' '{print $6, $1, $2, $3, $4, $5}' \
| sort \
> analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs

#Col names: transcript IDs, gene IDs, chr, start, end
!head analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs
!wc -l analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs

NM_001305288.1	105326593	NC_047559.1	17277556	17277558	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277568	17277570	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277625	17277627	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277631	17277633	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277647	17277649	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277650	17277652	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277700	17277702	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277734	17277736	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277761	17277763	NC_047559.1
NM_001305288.1	105326593	NC_047559.1	17277898	17277900	NC_047559.1
26098781 analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs


<h3> Join with annotations </h3>

In [32]:
#Join files to get GO annotations for DML
!join -1 1 -2 1 -t $'\t' \
analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs \
blast/Blastquery-GOslim.tab \
| sort \
| uniq \
> analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

uniq: write error


In [33]:
!head analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs.GOAnnot
!wc -l analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

XM_011413559.3	105317035	NC_047564.1	46227067	46227069	NC_047564.1	GO:0005777	peroxisome	other cytoplasmic organelle	C
XM_011413559.3	105317035	NC_047564.1	46227067	46227069	NC_047564.1	GO:0008445	D-aspartate oxidase activity	other molecular function	F
XM_011413559.3	105317035	NC_047564.1	46227067	46227069	NC_047564.1	GO:0046416	D-amino acid metabolic process	other metabolic processes	P
XM_011413559.3	105317035	NC_047564.1	46227126	46227128	NC_047564.1	GO:0005777	peroxisome	other cytoplasmic organelle	C
XM_011413559.3	105317035	NC_047564.1	46227126	46227128	NC_047564.1	GO:0008445	D-aspartate oxidase activity	other molecular function	F
XM_011413559.3	105317035	NC_047564.1	46227126	46227128	NC_047564.1	GO:0046416	D-amino acid metabolic process	other metabolic processes	P
XM_011413559.3	105317035	NC_047564.1	46227330	46227332	NC_047564.1	GO:0005777	peroxisome	other cytoplasmic organelle	C
XM_011413559.3	105317035	NC_047564.1	46227330	46227332	NC_047564.1	GO:0008445	D-aspartate oxidase act

<h1> Prepare for topGO enrichment analysis </h1>

In [1]:
cd /home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/analyses/GOterm-annotation/BOTH/

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/analyses/GOterm-annotation/BOTH


In [30]:
pwd

'/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/blast'

In [13]:
#Extract transcript IDs
#Filter and save unique IDs
!cut -f1 analyses/GOterm-annotation/BOTH/union_1x-BOTH.GeneIDs.geneOverlap.transcriptIDs.GOAnnot | uniq \
> analyses/GOterm-annotation/BOTH/union_1x-BOTH.transcriptIDs.unique

# !head union_1x.transcriptIDs.unique
!wc -l analyses/GOterm-annotation/BOTH/union_1x-BOTH.transcriptIDs.unique

27469 analyses/GOterm-annotation/BOTH/union_1x-BOTH.transcriptIDs.unique


In [10]:
cd /home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS/

/home/mngeorge/project-gigas_ploidy/bisulfide_analysis/WGBS


In [8]:
#Extract columns 2 and 3 to create list needed for topGO gene enrichment
#Convert ; to , for topGO formatting
!cut -f2,3 blast/_blast-annot.tab \
| tr ";" "," \
> blast/geneid2go.tab
!head blast/geneid2go.tab

XM_001305294.1	GO:0005576, GO:0007218
XM_001308866.2	GO:0005179, GO:0005576, GO:0006006
XM_034475035.1	GO:0005179, GO:0005576, GO:0006006
XM_011417422.3	GO:0005179, GO:0005576, GO:0006006
XM_011419035.3	GO:0005576
XM_011419037.3	GO:0005576
XM_011419038.3	GO:0005576
XM_011419039.3	GO:0005576
XM_020064360.2	GO:0005576
XM_011442798.3	GO:0005179, GO:0005576


In [16]:
#Filter the gene ID-GO file based on transcript IDs in 1x union data
!awk 'BEGIN { while(getline <"analyses/GOterm-annotation/BOTH/union_1x-BOTH.transcriptIDs.unique") id[$0]=1; } id[$1] ' blast/geneid2go.tab \
> analyses/GOterm-annotation/BOTH/geneid2go-union1x-BOTH.tab

!wc -l analyses/GOterm-annotation/BOTH/geneid2go-union1x-BOTH.tab

48263 analyses/GOterm-annotation/BOTH/geneid2go-union1x-BOTH.tab


<h1> Create Master Annotation table </h1>

In [2]:
!head GFF/GCF_902806645.1_cgigas_uk_roslin_v1_rna_from_genomic.fna

>lcl|NC_047559.1_ncrna_XR_004604272.1_1 [gene=LOC117693020] [db_xref=GeneID:117693020] [product=uncharacterized LOC117693020] [ncRNA_class=lncRNA] [transcript_id=XR_004604272.1] [location=join(9839..9922,10884..11386)] [gbkey=ncRNA]
TCTATAAATATCTGATGATTTGGTGTTTTATCATTCTTTTGATTGTGACTTTGTTGGATTATATGGGTTTTTCACTTGTT
AAAGATGGATATGGAAGATACGGAACAAACTTTTTCTGAAACGGTGTTGGAGAATTTCAACCTAGCCAAGGCGTACAATT
CTGTATCTGAAACACCACAACTCTTTTTAAGAAGGGATGATCGGATTTTTGGAAGATGTCTGACAGTCGATGAAGTAGAA
AGAGAAGTATGGTTGAGATATTCCTTTGACATTGATGGATGCGTTGTGGAGTTTAAATCTTTGGCCGCAATAAGAATGCC
CGTGAGACATATCACACTGTTTCCAAATGATAATGCAAGTCTACGGAACGTTAGAGGCTTTTTGAAAAAGCACGTCAGAT
TAACAGAATACAGTAAAAGCGAACTTATGACCTTCTCAAACAGCTTAGGGTTCCAGCTATTTTATGAAAGAGGGTGGGAA
TGTTGGATAGCTTTAATTCCTATGCACCAGATGCATTCAGTTGCATTATCCATTCAGCAAGTGAGACAACAAGCCCTCGC
CACATTTCTTCAGCTGAGAGAGAATTT
>lcl|NC_047559.1_mrna_XM_034463183.1_2 [gene=LOC109621113] [db_xref=GeneID:109621113] [product=uncharacterized LOC109621113] [transcript_id=XM_034463183.1] [location=join(1

In [4]:
#Convert to tab-delimited file
!perl -e '$count=0; $len=0; while(<>) {s/\r?\n//; s/\t/ /g; if (s/^>//) { if ($. != 1) {print "\n"} s/ |$/\t/; $count++; $_ .= "\t";} else {s/ //g; $len += length($_)} print $_;} print "\n"; warn "\nConverted $count FASTA records in $. lines to tabular format\nTotal sequence length: $len\n\n";' \
GFF/GCF_902806645.1_cgigas_uk_roslin_v1_rna_from_genomic.fna \
>GFF/GCF_902806645.1_cgigas_uk_roslin_v1_rna_from_genomic.tab
!head GFF/GCF_902806645.1_cgigas_uk_roslin_v1_rna_from_genomic.tab


Converted 73948 FASTA records in 2722600 lines to tabular format
Total sequence length: 208992633



In [6]:
#Isolate annotation
!cut -f2 GFF/GCF_902806645.1_cgigas_uk_roslin_v1_rna_from_genomic.tab \
| tr "[]" "\t" \
| cut -f6,8 \
| tr "=" "\t" \
| cut -f2 \
> GFF/cgigas_uk_roslin_v1_rna_from_genomic_annotation.tab

In [7]:
#Isolate transcript ID
!cut -f2 GFF/GCF_902806645.1_cgigas_uk_roslin_v1_rna_from_genomic.tab \
| tr "[]" "\t" \
| cut -f6,8 \
| tr "=" "\t" \
| cut -f4 \
> GFF/cgigas_uk_roslin_v1_rna_from_genomic_transcript.tab

In [8]:
#Paste annotation and transcript together
#Extract lines with a transcript ID (XM)
#Sort
#Save output
!paste GFF/cgigas_uk_roslin_v1_rna_from_genomic_transcript.tab GFF/cgigas_uk_roslin_v1_rna_from_genomic_annotation.tab \
| grep "XM" \
| sort \
> GFF/cgigas_uk_roslin_v1_rna_from_genomic_annot.transcript.tab

In [9]:
!head GFF/cgigas_uk_roslin_v1_rna_from_genomic_annot.transcript.tab
!wc -l GFF/cgigas_uk_roslin_v1_rna_from_genomic_annot.transcript.tab

XM_011413559.3	D-aspartate oxidase, transcript variant X2
XM_011413566.3	uncharacterized LOC105317040, transcript variant X1
XM_011413583.3	PDZ and LIM domain protein Zasp, transcript variant X1
XM_011413585.3	PDZ and LIM domain protein Zasp, transcript variant X2
XM_011413589.3	zinc finger and BTB domain-containing protein 11, transcript variant X1
XM_011413590.3	zinc finger and BTB domain-containing protein 11, transcript variant X3
XM_011413596.3	uncharacterized LOC105317059
XM_011413597.3	uncharacterized LOC105317060
XM_011413601.3	mitogen-activated protein kinase HOG1, transcript variant X1
XM_011413602.3	mitogen-activated protein kinase HOG1, transcript variant X2
62377 GFF/cgigas_uk_roslin_v1_rna_from_genomic_annot.transcript.tab


<h1> All DML </h1>

In [ ]:
!head ../../output/10_DML-characterization/DML-pH-50-Cov5-All-Gene-wb.bed

In [ ]:
#Isolate column with gene ID information
#Convert =, :, and ; to \t
#Isolate gene ID

!cut -f13 ../../output/10_DML-characterization/DML-pH-50-Cov5-All-Gene-wb.bed \
| tr "=:;" "\t" \
| cut -f5 \
> DML-pH-50-Cov5-All.GeneIDs

!head DML-pH-50-Cov5-All.GeneIDs
!wc -l DML-pH-50-Cov5-All.GeneIDs

In [ ]:
#Join files to get transcript ID for DML
!join -1 1 -2 1 -t $'\t' \
DML-pH-50-Cov5-All.GeneIDs.geneOverlap \
cgigas_uk_roslin_v1_mRNA.transcriptID.geneID \
| uniq | awk -F'\t' -v OFS='\t' '{print $6, $1, $2, $3, $4, $5}' \
| sort \
> DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs

In [ ]:
#Col names: transcript IDs, gene IDs, chr, start, end
!head DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs
!wc -l DML-pH-50-Cov5-All.GeneIDs.geneOverlap.trans

In [ ]:
#Join files to get GO annotations for DML
!join -1 1 -2 1 -t $'\t' \
DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs \
Blastquery-GOslim.tab \
| uniq \
| sort \
> DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

!head DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs.GOAnnot
!wc -l DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs.GOAnnot

In [ ]:
#Join files to get product names for genes with DML
!join -1 1 -2 1 -t $'\t' \
DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs.GOAnnot \
cgigas_uk_roslin_v1_rna_from_genomic_annot.transcript.tab \
| uniq \
| sort \
> DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs.GOAnnot.productID

!head DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs.GOAnnot.productID
!wc -l DML-pH-50-Cov5-All.GeneIDs.geneOverlap.transcriptIDs.GOAnnot.productID